In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,7"

import torch
print("GPUs available:", torch.cuda.device_count())
print("Flash-attn loaded:", "flash_attn" in dir(__import__("flash_attn")))


GPUs available: 4
Flash-attn loaded: False


In [2]:
import sys
print("Python executable:", sys.executable)

import torch
print("Torch version:", torch.__version__, "CUDA available:", torch.cuda.is_available())

try:
    import flash_attn
    print("Flash-attn loaded:", flash_attn.__version__)
except Exception as e:
    print("Flash-attn import error:", e)


Python executable: /opt/conda/bin/python
Torch version: 2.2.2 CUDA available: True
Flash-attn loaded: 2.6.3


In [3]:
# ===============================================
# Notebook Cell: 环境 & GPU & flash-attn 检查
# ===============================================

import sys
import subprocess
import torch

print("=== Python ===")
print("Executable:", sys.executable)
print("Version   :", sys.version)

print("\n=== PyTorch & CUDA ===")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}, Memory Allocated: {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

print("\n=== Flash-Attention ===")
try:
    import flash_attn
    print("Flash-attn loaded:", flash_attn.__version__)
except Exception as e:
    print("Flash-attn import error:", e)
    print("Possible reasons:")
    print("1) flash-attn not installed in this Python environment")
    print("2) CUDA/nvcc not available when compiled")
    print("3) Torch version mismatch")
    
    print("\n=== Quick Fix Command (run in container shell) ===")
    print("python -m pip install --upgrade --force-reinstall packaging ninja einops")
    print("MAX_JOBS=64 python -m pip install flash-attn==2.6.3 --no-build-isolation")
    print("\nMake sure you use a devel PyTorch image and container has nvcc available!")


=== Python ===
Executable: /opt/conda/bin/python
Version   : 3.10.14 (main, Mar 21 2024, 16:24:04) [GCC 11.2.0]

=== PyTorch & CUDA ===
Torch version: 2.2.2
CUDA available: True
CUDA devices count: 4
Device 0: NVIDIA L40S, Memory Allocated: 0.00 GB
Device 1: NVIDIA L40S, Memory Allocated: 0.00 GB
Device 2: NVIDIA L40S, Memory Allocated: 0.00 GB
Device 3: NVIDIA L40S, Memory Allocated: 0.00 GB

=== Flash-Attention ===
Flash-attn loaded: 2.6.3


In [5]:
# ===============================================
# Notebook: Time-MoE Benchmark & Training Template
# ===============================================

# Step 0: 环境检查
import sys
import torch

print("=== Python ===")
print("Executable:", sys.executable)
print("Version   :", sys.version)

print("\n=== PyTorch & CUDA ===")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}, Memory Allocated: {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

print("\n=== Flash-Attention ===")
try:
    import flash_attn
    print("Flash-attn loaded:", flash_attn.__version__)
except Exception as e:
    print("Flash-attn import error:", e)
    print("⚠️ 建议检查 Python 环境和 CUDA / nvcc 是否可用。")

# ===================================================
# Step 1: 设置路径 & CUDA 可见 GPU
# ===================================================
import os

# 数据路径
DATASET_DIR = "/workspace/dataset"
TIME300B_DIR = "/workspace/Time-300B"

# 输出 / 模型路径
OUTPUT_DIR = "/workspace/outputs"

# 指定使用 GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  # 你当前四块 L40S GPU

# ===================================================
# Step 2: 评估 benchmark
# ===================================================
# 可以修改 eval_dataset 指定数据集
eval_dataset = "ETTh1"       # ETTh1 / ETTh2 / ETTm1 / Weather
eval_file = os.path.join(DATASET_DIR, f"{eval_dataset}.csv")
prediction_length = 96        # 根据论文设置

print(f"Running evaluation on {eval_dataset}...")

!python /workspace/run_eval.py -d {eval_file} -p {prediction_length}

# 输出会在 logs/ 或 outputs/ 目录下生成评估结果

# ===================================================
# Step 3: 加载预训练模型进行推理
# ===================================================
import torch
from transformers import AutoModelForCausalLM

context_length = 12
seqs = torch.randn(2, context_length).to("cuda")  # [batch_size, context_length]

# 使用 flash-attn 加速
model = AutoModelForCausalLM.from_pretrained(
    'Maple728/TimeMoE-50M',
    device_map="auto",
    attn_implementation='flash_attention_2',
    trust_remote_code=True,
)

# 归一化
mean, std = seqs.mean(dim=-1, keepdim=True), seqs.std(dim=-1, keepdim=True)
normed_seqs = (seqs - mean) / std

# 预测
prediction_length = 6
output = model.generate(normed_seqs, max_new_tokens=prediction_length)  # [batch_size, context+pred]
normed_predictions = output[:, -prediction_length:]
predictions = normed_predictions * std + mean

print("Sample predictions:", predictions)

# ===================================================
# Step 4: Multi-GPU Fine-tuning
# ===================================================
# 单节点多 GPU 训练
# 使用 torch_dist_run.py 自动分配多 GPU
train_data_path = os.path.join(DATASET_DIR, "YourDataset.jsonl")  # 替换为实际训练数据

!python /workspace/torch_dist_run.py /workspace/main.py -d {train_data_path} --from_scratch --stride 1

# 训练完成后，模型将保存在 OUTPUT_DIR/logs/time_moe 下
print(f"Training finished. Check {OUTPUT_DIR}/logs/time_moe for saved models.")

# ===================================================
# Step 5: 保存评估与训练日志
# ===================================================
import shutil

log_src = os.path.join(OUTPUT_DIR, "logs/time_moe")
log_dst = "/workspace/time_moe_results"
shutil.copytree(log_src, log_dst, dirs_exist_ok=True)
print(f"All results copied to {log_dst}")


=== Python ===
Executable: /opt/conda/bin/python
Version   : 3.10.14 (main, Mar 21 2024, 16:24:04) [GCC 11.2.0]

=== PyTorch & CUDA ===
Torch version: 2.2.2
CUDA available: True
CUDA devices count: 4
Device 0: NVIDIA L40S, Memory Allocated: 0.00 GB
Device 1: NVIDIA L40S, Memory Allocated: 0.00 GB
Device 2: NVIDIA L40S, Memory Allocated: 0.00 GB
Device 3: NVIDIA L40S, Memory Allocated: 0.00 GB

=== Flash-Attention ===
Flash-attn loaded: 2.6.3
Running evaluation on ETTh1...
/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
config.json: 100%|█████████████████████████████| 891/891 [00:00<00:00, 3.00MB/s]
model.safetensors: 100%|█████████████████████| 227M/227M [00:03<00:00, 64.5MB/s]
generation_config.json: 100%|█████████████████| 69.0/69.0 [00:00<00:00, 841kB/

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/Maple728/TimeMoE-50M:
- configuration_time_moe.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/Maple728/TimeMoE-50M:
- ts_generation_mixin.py
. Make sure to double-check they do not contain any added malicio

Sample predictions: tensor([[-0.5207, -0.1987, -0.2402, -0.2409, -0.2167, -0.2275],
        [-0.2593, -0.4568, -0.4589, -0.4752, -0.4884, -0.4724]],
       device='cuda:0')
torchrun --master_addr=localhost --master_port=9899 --node_rank=0 --nproc_per_node=4 --nnodes=1 /workspace/main.py -d /workspace/dataset/YourDataset.jsonl --from_scratch --stride 1
[2026-01-23 06:03:53,693] torch.distributed.run: [WARNING] 
[2026-01-23 06:03:53,693] torch.distributed.run: [WARNING] *****************************************
[2026-01-23 06:03:53,693] torch.distributed.run: [WARNING] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
[2026-01-23 06:03:53,693] torch.distributed.run: [WARNING] *****************************************
2026-01-23 06:03:56,959 - log_util.py[pid:2229181;line:52:log_in_local_rank_0] - INFO: Set global_batch_size to 64


FileNotFoundError: [Errno 2] No such file or directory: '/workspace/outputs/logs/time_moe'

In [6]:
# %% [markdown]
# ## 测试加载 ETTh1 或 Time-300B 数据

# %%
import os
import pandas as pd
import numpy as np
import json

# --- 配置路径 ---
use_time300b = False  # True 切换到 Time-300B
etth1_path = "../dataset/ETT-small/ETTh1.csv"

time300b_dir = "Time-300B/energy/electricity_15min"  # 选定子集
time300b_bin = os.path.join(time300b_dir, "data-1-of-1.bin")
time300b_meta = os.path.join(time300b_dir, "meta.json")

# --- 加载数据 ---
if use_time300b:
    # 读取 meta
    with open(time300b_meta, "r") as f:
        meta = json.load(f)
    print("Time-300B meta:", meta)

    # 假设 float32 二进制
    data = np.fromfile(time300b_bin, dtype=np.float32)
    
    # reshape 根据 meta info
    seq_len = meta.get("seq_len", -1)
    num_series = meta.get("num_series", -1)
    if seq_len > 0 and num_series > 0:
        data = data.reshape(num_series, seq_len)
    
    print("Time-300B 数据形状:", data.shape)
else:
    df = pd.read_csv(etth1_path)
    print("ETTh1 前 5 行:\n", df.head())
    print("ETTh1 数据形状:", df.shape)

# %% [markdown]
# 可以根据需要在后续 cell 做训练或特征处理


FileNotFoundError: [Errno 2] No such file or directory: 'dataset/ETT-small/ETTh1.csv'

In [8]:
import pandas as pd

etth1_path = "../dataset/ETT-small/ETTh1.csv"  # 相对于当前目录
df = pd.read_csv(etth1_path)
print(df.head())


                  date   HUFL   HULL   MUFL   MULL   LUFL   LULL         OT
0  2016-07-01 00:00:00  5.827  2.009  1.599  0.462  4.203  1.340  30.531000
1  2016-07-01 01:00:00  5.693  2.076  1.492  0.426  4.142  1.371  27.787001
2  2016-07-01 02:00:00  5.157  1.741  1.279  0.355  3.777  1.218  27.787001
3  2016-07-01 03:00:00  5.090  1.942  1.279  0.391  3.807  1.279  25.044001
4  2016-07-01 04:00:00  5.358  1.942  1.492  0.462  3.868  1.279  21.948000


In [9]:
# ===================================================
# Step 1: 设置路径 & CUDA 可见 GPU
# ===================================================
import os

# 自动检测当前工作目录（假设 notebook 在项目根目录下）
PROJECT_ROOT = os.getcwd()  # e.g. /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD

# 数据路径
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset")
TIME300B_DIR = os.path.join(PROJECT_ROOT, "Time-300B")

# 输出 / 模型路径
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")

print("Project root:", PROJECT_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Time-300B dir:", TIME300B_DIR)
print("Output dir:", OUTPUT_DIR)

# 指定使用 GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  # 你当前四块 L40S GPU



Project root: /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD/notebooks
Dataset dir: /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD/notebooks/dataset
Time-300B dir: /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD/notebooks/Time-300B
Output dir: /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD/notebooks/outputs


In [13]:
# ===================================================
# Step 1: 设置路径 & CUDA 可见 GPU
# ===================================================
import os

# 获取 notebook 上一级目录的上一级作为项目根
# PROJECT_ROOT = os.path.dirname(os.path.dirname(os.getcwd()))  
PROJECT_ROOT = "/beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD/"
# 如果 notebook 在 /.../WARP-MoE_26KDD/notebooks/，PROJECT_ROOT 就是 /.../WARP-MoE_26KDD

# 数据路径
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset")
TIME300B_DIR = os.path.join(PROJECT_ROOT, "Time-300B")

# 输出 / 模型路径
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")

print("Project root:", PROJECT_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Time-300B dir:", TIME300B_DIR)
print("Output dir:", OUTPUT_DIR)

# 指定使用 GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  # 当前四块 L40S GPU

# ===================================================
# Step 2: 评估 benchmark
# ===================================================
eval_dataset = "ETTh1"       # ETTh1 / ETTh2 / ETTm1 / Weather
eval_file = os.path.join(DATASET_DIR, "ETT-small", f"{eval_dataset}.csv")  # 这里指定 ETT-small 子目录
prediction_length = 96        # 根据论文设置

print(f"Running evaluation on {eval_dataset}...")
!python {os.path.join(PROJECT_ROOT, "run_eval.py")} -d {eval_file} -p {prediction_length}

# ===================================================
# Step 4: Multi-GPU Fine-tuning
# ===================================================
# 单节点多 GPU 训练
train_data_path = os.path.join(DATASET_DIR, "YourDataset.jsonl")  # 替换为实际训练数据

!python {os.path.join(PROJECT_ROOT, "torch_dist_run.py")} {os.path.join(PROJECT_ROOT, "main.py")} -d {train_data_path} --from_scratch --stride 1

print(f"Training finished. Check {os.path.join(OUTPUT_DIR, 'logs/time_moe')} for saved models.")

# ===================================================
# Step 5: 保存评估与训练日志
# ===================================================
import shutil

log_src = os.path.join(OUTPUT_DIR, "logs/time_moe")
log_dst = os.path.join(PROJECT_ROOT, "time_moe_results")
shutil.copytree(log_src, log_dst, dirs_exist_ok=True)
print(f"All results copied to {log_dst}")


Project root: /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD/
Dataset dir: /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD/dataset
Time-300B dir: /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD/Time-300B
Output dir: /beacon/data01/chengjie.zheng001/Projects/09PseudoTimeMoE/26KDD/WARP-MoE_26KDD/outputs
Running evaluation on ETTh1...
/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-01-23 06:35:05,792 - log_util.py[pid:2232475;line:52:log_in_local_rank_0] - INFO: >>> Split test data from 2017-10-02 16:00:00 to 2018-02-20 23:00:00, and evaluation start date is: 2017-10-24 00:00:00
100%|█████████████████████████████████████████| 609/609 [02:06<00:00,  4.83it/s]
0 